### 00 Data Dictionary

#### Purpose
- Download the official data dictionary of every source (OpenFEMA field definitions and Census ACS variable labels).
- Save each one to `docs/data_dictionary/` as an Excel file, so notebook 02 can check the data against it.
- Confirm that every column in the raw data is described in its dictionary.

#### Outputs
- `fema_<dataset>_v<version>.xlsx` (one per FEMA dataset)
- `census_acs5_2022_selected.xlsx`

`Data Dictionary Compilation.xlsx` in the same folder is compiled by hand and is not produced by this notebook.

### Section 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

# Find the repo root from any folder, so `src` can be imported
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd
import requests
import duckdb

from src.paths import DICTIONARIES, RAW_FEMA, ensure_dirs
from src.download import ACS_VARS, ACS_YEAR   # one list of ACS variables, shared with notebook 01

ensure_dirs()
PULLED_ON = pd.Timestamp.today().date()       # FEMA updates the dictionaries; record when we pulled them

### Section 1. FEMA field definitions

In [2]:
# 1. FEMA field definitions (OpenFEMA dataset-fields API)
FEMA_URL = "https://www.fema.gov/api/open/v1/OpenFemaDataSetFields"
FEMA_DATASETS = {
    "NfipPolicies": 3,
    "NfipClaims": 3,
    "DisasterDeclarationsSummaries": 2,
    "NfipCommunityStatusBook": 1,
}
COLUMNS = ["openFemaDataSet", "datasetVersion", "sortOrder", "name", "title",
           "type", "isNullable", "primaryKey", "description"]

frames = []
for dataset, version in FEMA_DATASETS.items():
    params = {
        "$filter": f"openFemaDataSet eq '{dataset}' and datasetVersion eq {version}",
        "$top": 1000,
        "$format": "json",
    }
    r = requests.get(FEMA_URL, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["OpenFemaDataSetFields"])[COLUMNS].sort_values("sortOrder")
    df["pulled_on"] = PULLED_ON
    df.to_excel(DICTIONARIES / f"fema_{dataset}_v{version}.xlsx", index=False)
    print(f"{dataset}: {len(df)} fields")
    frames.append(df)

fema_dict = pd.concat(frames, ignore_index=True)

NfipPolicies: 87 fields
NfipClaims: 84 fields
DisasterDeclarationsSummaries: 29 fields
NfipCommunityStatusBook: 18 fields


### Section 2. Census ACS variables used in this project

In [3]:
# 2. Census ACS variable labels, saved with the same columns as the FEMA files
#    (name, title, type, description), so notebook 02 can treat every dictionary the same way
r = requests.get(f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5/variables.json", timeout=300)
r.raise_for_status()
variables = r.json()["variables"]

acs_dict = pd.DataFrame([
    {"name": v,
     "readable_name": readable,
     "title": variables[v]["label"],
     "type": variables[v].get("predicateType"),
     "description": f'{variables[v]["concept"]}: {variables[v]["label"]}',
     "pulled_on": PULLED_ON}
    for v, readable in ACS_VARS.items()
])
acs_dict.to_excel(DICTIONARIES / f"census_acs5_{ACS_YEAR}_selected.xlsx", index=False)
acs_dict

,name,readable_name,title,type,description,pulled_on
0,B19013_001E,median_household_income,Estimate!!Median household income in the past ...,int,Median Household Income in the Past 12 Months ...,2026-11-12
1,B25077_001E,median_home_value,Estimate!!Median value (dollars),int,Median Value (Dollars): Estimate!!Median value...,2026-11-12
2,B01002_001E,median_age,Estimate!!Median age --!!Total:,float,Median Age by Sex: Estimate!!Median age --!!To...,2026-11-12
3,B25003_001E,occupied_housing_units,Estimate!!Total:,int,Tenure: Estimate!!Total:,2026-11-12
4,B25003_002E,owner_occupied_units,Estimate!!Total:!!Owner occupied,int,Tenure: Estimate!!Total:!!Owner occupied,2026-11-12
5,B01003_001E,total_population,Estimate!!Total,int,Total Population: Estimate!!Total,2026-11-12


### Section 3. Cancellation reason codes

In [4]:
# 3. Show the cancellation reason codes (used to build the lapse target)
codes = fema_dict.query(
    "openFemaDataSet == 'NfipPolicies' and name == 'cancellationVoidanceReasonCode'"
)["description"].iloc[0]
print(codes)

Reason code indicating why a policy was cancelled/nullified.
Legal values (value : description):
01 : Building Sold or Removed;
02 : Contents Sold or Removed (contents-only policy);
03 : Policy Canceled and Rewritten to Establish a Common Expiration; Date with Other Lines of Insurance;
04 : Duplicate Policies (NFIP);
05 : Non-Payment;
06 : Ineligible Risk;
08 : No Insurable Interest;
09 : PMR (Physical Map Revision) or LOMR ( Letter of Map Revision) with Lender Release;
10 : Closed Basin Lake (ineligible);
16 : FEMA Letter of Determination with Lender Release;
17 : Duplicate Policy from Source Other than the NFIP (Legacy System);
20 : LOMA (Letter of Map Amendment) with Lender Release;
21 : SRL (Severe Repetitive Loss) Written with Incorrect Insurer;
22 : Cancel/Rewrite Due to Misrating;
23 : Fraud;
24 : Cancel/Rewrite to PRP after Map Revision;
25 : Cancel/Rewrite to Process HFIAA Refund;
26 : Duplicate Policy from Source Other than the NFIP;
27 : Property becomes ineligible for cover

### Section 4. Check the dictionaries against the data
Every column in a raw file should be described in its dictionary, and the other way round. A column found in only one of the two needs a note before cleaning.

In [5]:
# 4. Compare the dictionary fields with the columns of the raw Parquet files
files = {"NfipPolicies": "nfip_policies_v3", "NfipClaims": "nfip_claims_v3",
         "DisasterDeclarationsSummaries": "disaster_declarations_summaries_v2",
         "NfipCommunityStatusBook": "nfip_community_status_book_v1"}

rows = []
for ds, f in files.items():
    data_cols = {r[0] for r in duckdb.sql(f"DESCRIBE SELECT * FROM '{RAW_FEMA / f}.parquet'").fetchall()}
    dict_cols = set(fema_dict.loc[fema_dict["openFemaDataSet"] == ds, "name"])
    rows.append({"dataset": ds, "columns_in_data": len(data_cols), "fields_in_dictionary": len(dict_cols),
                 "only_in_data": sorted(data_cols - dict_cols),
                 "only_in_dictionary": sorted(dict_cols - data_cols)})

pd.DataFrame(rows)

,dataset,columns_in_data,fields_in_dictionary,only_in_data,only_in_dictionary
0,NfipPolicies,87,87,[],[]
1,NfipClaims,84,84,[],[]
2,DisasterDeclarationsSummaries,29,29,[],[]
3,NfipCommunityStatusBook,18,18,[],[]
